###  Predicting New York City Taxi Fares based in Input Features using Apache Spark

#### **Dataset: NYC Taxi Trip Data**

- **Description**: This dataset includes trip details such as pickup/drop-off locations, times, distances, passenger counts, and fares.
- **Size**: Several gigabytes (depends on the selected month/year).
- **Source**: The NYC Taxi & Limousine Commission (TLC) provides this data for free.
- **Download**: The data can be downloaded from [NYC TLC Trip Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

#### **1. Setting Up Azure Databricks**
1. **Create a Databricks Workspace**:
   - In the Azure portal, create an Azure Databricks workspace.
   - Attach it to a "Pay-as-You-Go" or free-tier resource group.
   - Launch the workspace and set up a cluster (single node for small-scale demos).

2. **Upload NYC Taxi Data to the Lakehouse**:
   - Download a subset of the NYC taxi trip dataset (e.g., January 2023 in CSV format).
   - Upload the data to your Databricks Lakehouse.

#### **2. Data Exploration**
Introduce your students to big data exploration using Spark. Use the following steps:

##### a. Load the Dataset

In [ ]:
# Load the dataset from the Lakehouse
file_path = "/mnt/lakehouse/taxi_trip_data.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)
df.printSchema()
df.show(5)

: 

*Explanation*: This loads the dataset into a Spark DataFrame and displays its schema and a few rows to understand the structure.

##### b. Perform Basic Analysis

In [ ]:
# Count the number of rows in the dataset
print(f"Total Rows: {df.count()}")

# Display distinct values for key columns
df.select("payment_type").distinct().show()

# Calculate basic statistics for numerical columns
df.describe(["trip_distance", "fare_amount"]).show()

*Explanation*: This shows the size of the dataset, distinct values in categorical columns, and basic statistics for numerical features.

#### **3. Data Cleaning**
Demonstrate how to clean and preprocess big data for machine learning.

##### a. Handle Missing Values

In [ ]:
# Remove rows with missing values
cleaned_df = df.dropna()

# Show the number of rows after cleaning
print(f"Rows after cleaning: {cleaned_df.count()}")

##### b. Filter Outliers

In [ ]:
# Remove trips with unrealistic distances or fares
filtered_df = cleaned_df.filter((cleaned_df["trip_distance"] > 0) & 
                                (cleaned_df["fare_amount"] > 0))

filtered_df.show(5)

*Explanation*: Cleaning ensures the data quality required for accurate machine learning models.

#### **4. Feature Engineering**
Introduce students to feature engineering concepts like one-hot encoding and vectorization.

##### a. Extract Features for ML

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Index and encode categorical columns
payment_indexer = StringIndexer(inputCol="payment_type", outputCol="payment_type_index")
encoded_df = payment_indexer.fit(filtered_df).transform(filtered_df)

# Assemble features into a vector
assembler = VectorAssembler(
    inputCols=["trip_distance", "fare_amount", "payment_type_index"],
    outputCol="features"
)
final_df = assembler.transform(encoded_df).select("features", encoded_df["fare_amount"].alias("label"))
final_df.show(5)

*Explanation*: Features like `trip_distance`, `fare_amount`, and `payment_type` are transformed into a vector column suitable for Spark MLlib models.

#### **5. Train a Machine Learning Model**
Demonstrate training a regression model (e.g., predict fare amounts).

##### a. Split Data into Training and Test Sets

In [ ]:
train_data, test_data = final_df.randomSplit([0.8, 0.2], seed=42)
train_data.describe().show()
test_data.describe().show()

##### b. Train a Linear Regression Model

In [ ]:
from pyspark.ml.regression import LinearRegression

# Initialize the model
lr = LinearRegression(featuresCol="features", labelCol="label")

# Train the model
lr_model = lr.fit(train_data)

# Print model coefficients
print(f"Coefficients: {lr_model.coefficients}")
print(f"Intercept: {lr_model.intercept}")

*Explanation*: A linear regression model predicts the fare amount based on the input features.

#### **6. Evaluate the Model**
Evaluate the model's performance using metrics like RMSE.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

# Make predictions
predictions = lr_model.transform(test_data)

# Evaluate the model
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE): {rmse}")

# Show predictions
predictions.select("label", "prediction").show(5)

*Explanation*: RMSE measures the average error in predictions, which shows model accuracy.

#### **7. Visualization with Power BI**
Demonstrate how to visualize big data insights using Power BI:

1. Save results to the Lakehouse:

In [ ]:
predictions.write.csv("/mnt/lakehouse/predictions.csv", header=True)
# Save the model
lr_model.save("/mnt/lakehouse/taxi_fare_model")

2. Import the predictions CSV into Power BI.
3. Create visualizations such as:
   - Scatter plots of actual vs. predicted fare amounts.
   - Bar charts showing average fare per payment type.

### **Outcome**
- Understand how to use Databricks for big data processing.
- Learn Spark SQL and Spark MLlib for machine learning workflows.
- Gain hands-on experience with feature engineering, model training, and evaluation.
- Visualize results using Power BI.